In [ ]:
import subprocess, sys, os

os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet", "hf_transfer"], capture_output=True)
for pkg, extra in [
    ("tokenizers==0.21.0",   []),
    ("transformers==4.49.0", ["--no-deps"]),
    ("fast-disambig",        []),
    ("xformers",             ["--no-deps"]),
    ("scikit-learn",         []),
]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg] + extra, check=True)
    print(f"{pkg}")
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"], capture_output=True)
print("hf_xet force-removed")
print(" Runtime → Restart session, then run CELL 2")

In [ ]:
import subprocess, sys, os
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "hf_xet"], capture_output=True)

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["HF_HUB_DISABLE_XET"] = "1"

from huggingface_hub import login
from transformers import AutoModel, AutoTokenizer
import torch, json, time, copy
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim
from getpass import getpass

HF_TOKEN = getpass("Enter your Hugging Face token: ")
login(token=HF_TOKEN)

device    = "cuda" if torch.cuda.is_available() else "cpu"
print(f"using: {device}")
tokenizer = AutoTokenizer.from_pretrained("AhmadAfles/my-first-AraGenre", trust_remote_code=True)
encoder   = AutoModel.from_pretrained("AhmadAfles/my-first-AraGenre", trust_remote_code=True, torch_dtype=torch.float32).to(device)
encoder.eval()
for param in encoder.parameters():
    param.requires_grad = False

def encode(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=512, padding=True).to(device)
    with torch.no_grad():
        out = encoder(**inputs)
    mask = inputs["attention_mask"].unsqueeze(-1).float()
    emb  = torch.sum(out.last_hidden_state * mask, 1) / torch.clamp(mask.sum(1), min=1e-9)
    return F.normalize(emb, p=2, dim=1).squeeze(0)


Upload training data

In [ ]:

from google.colab import files as colab_files

print("upload training examples:")
up1 = colab_files.upload(); texts_file = list(up1.keys())[0]

print("upload training genres:")
up2 = colab_files.upload(); defs_file = list(up2.keys())[0]

texts_data = json.load(open(texts_file, encoding="utf-8"))

genre_defs_raw = json.load(open(defs_file, encoding="utf-8"))

GENRE_DEFS = {
    entry["specific_genre"]: entry["specific_genre_definition_ar"]
    for entry in genre_defs_raw
}

SPECIFIC_TO_BROAD = {
    entry["specific_genre"]: entry["broad_genre"]
    for entry in genre_defs_raw
}

print(f"\n✅ {len(GENRE_DEFS)}specific genres| {len(texts_data)} examples")

In [ ]:

from collections import defaultdict
import random
random.seed(42)

texts_by_genre = defaultdict(list)
for item in texts_data:
    texts_by_genre[item["specific_genre"]].append(item["Text"])

TRAINING_TEXTS = {}
for genre in GENRE_DEFS:
    all_texts = texts_by_genre.get(genre, [])
    random.shuffle(all_texts)
    TRAINING_TEXTS[genre] = all_texts

GENRE_LIST = list(GENRE_DEFS.keys())

In [ ]:

class DefinitionAdapter(nn.Module):
    def __init__(self, dim=768, hidden=128):
        super().__init__()
        self.W1 = nn.Linear(dim, hidden)
        self.W2 = nn.Linear(hidden, dim)

    def forward(self, d):
        correction = self.W2(torch.relu(self.W1(d)))
        return F.normalize(d + correction, p=2, dim=-1)

adapter = DefinitionAdapter().to(device)
n_params = sum(p.numel() for p in adapter.parameters())
print(f"Training parameters: {n_params:,}")

TEXT_VECTORS = {g: [encode(t) for t in texts] for g, texts in TRAINING_TEXTS.items()}

DEF_RAW_VECTORS = {g: encode(d) for g, d in GENRE_DEFS.items()}


In [ ]:


def focal_loss(logits, target_idx, gamma=2.0):

    log_probs = F.log_softmax(logits, dim=-1)
    probs     = log_probs.exp()
    p_correct = probs[0, target_idx]
    ce = F.nll_loss(log_probs, torch.tensor([target_idx], device=logits.device))
    focal_weight = (1 - p_correct) ** gamma
    return focal_weight * ce


GAMMA       = 2.0
TEMPERATURE = 0.1
NUM_EPOCHS  = 30
optimizer   = optim.Adam(adapter.parameters(), lr=1e-3)

history = []


for epoch in range(1, NUM_EPOCHS + 1):
    adapter.train()
    epoch_loss, n_steps = 0.0, 0
    training_pairs = [(g, v) for g, vecs in TEXT_VECTORS.items() for v in vecs]
    random.shuffle(training_pairs)

    for correct_genre, text_vec in training_pairs:
        prototypes = torch.stack([adapter(DEF_RAW_VECTORS[g]) for g in GENRE_LIST])
        scores = torch.matmul(prototypes, text_vec) / TEMPERATURE
        correct_idx = GENRE_LIST.index(correct_genre)

        loss = focal_loss(scores.unsqueeze(0), correct_idx, gamma=GAMMA)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        n_steps += 1

    avg_loss = epoch_loss / n_steps
    history.append({"epoch": epoch, "loss": avg_loss})
    print(f"Epoch {epoch:2d}/{NUM_EPOCHS} | loss: {avg_loss:.4f}")

adapter.eval()


In [ ]:
from google.colab import files as colab_files
import json
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

print("genre definitions")
up1 = colab_files.upload(); genre_defs_file = list(up1.keys())[0]

print("examples")
up2 = colab_files.upload(); examples_file = list(up2.keys())[0]



genre_definitions = json.load(open(genre_defs_file, encoding="utf-8"))
examples        = json.load(open(examples_file, encoding="utf-8"))


In [ ]:

SPECIFIC_GENRE_ENCODED_DEFS = {}
SPECIFIC_TO_BROAD_GENRE_MAP = {}
SPECIFIC_GENRE_LIST = []

for entry in genre_definitions:
    specific_genre_name = entry["specific_genre"]
    specific_genre_def_text = entry["specific_genre_definition"]
    broad_genre_name = entry["broad_genre"]

    SPECIFIC_GENRE_ENCODED_DEFS[specific_genre_name] = encode(specific_genre_def_text)
    SPECIFIC_TO_BROAD_GENRE_MAP[specific_genre_name] = broad_genre_name
    SPECIFIC_GENRE_LIST.append(specific_genre_name)

# Ensure the order is consistent for later metric calculations
SPECIFIC_GENRE_LIST.sort()


In [ ]:
# Run prediction loop

all_gold_broad_genres = []
all_gold_specific_genres = []
all_pred_broad_genres = []
all_pred_specific_genres = []
misclassified_examples = []


# Prepare prototypes for all specific genres
adapter.eval()
with torch.no_grad():
    prototypes = {
        g: adapter(SPECIFIC_GENRE_ENCODED_DEFS[g]).cpu()
        for g in SPECIFIC_GENRE_LIST
    }

    for example in examples:
        example_id = example['id']
        text = example['text']

        # Encode text and apply adapter
        text_vec = encode(text).cpu()

        # Calculate similarities to all specific genre prototypes
        sims = {g: torch.dot(text_vec, p).item() for g, p in prototypes.items()}

        # Get predicted specific genre
        predicted_specific = max(sims, key=sims.get)
        predicted_broad = SPECIFIC_TO_BROAD_GENRE_MAP[predicted_specific]
